In [11]:
!pip install kagglehub --upgrade


In [ ]:
from google.colab import files
files.upload()  


Saving kaggle (2).json to kaggle (2) (1).json


{'kaggle (2) (1).json': b'{"username":"rozhinajahdi","key":"cdb8887dd508eb17b8229d3b0030ff64"}'}

In [13]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


mv: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
import kagglehub


dataset_path = kagglehub.dataset_download("walidguirat/office-object-detection")

print("Dataset path:", dataset_path)


Dataset path: /kaggle/input/office-object-detection


In [15]:
!ls /kaggle/input/office-object-detection


downloaded_images


In [ ]:


import os
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from PIL import Image
import pandas as pd
from sklearn.model_selection import train_test_split


class OfficeObjectsDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
       
        self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        self.images = []
        self.labels = []

        for cls in self.classes:
            cls_dir = os.path.join(root_dir, cls)
            for img_name in os.listdir(cls_dir):
                if img_name.endswith(('.jpg', '.jpeg', '.png')):
                    self.images.append(os.path.join(cls_dir, img_name))
                    self.labels.append(self.class_to_idx[cls])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


data_dir = "/kaggle/input/office-object-detection/downloaded_images"


if not os.path.exists(data_dir):
    raise FileNotFoundError(f"Dataset directory not found: {data_dir}. Please verify the path.")

print("Dataset contents:", os.listdir(data_dir))


train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomCrop(64, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


train_full_dataset = OfficeObjectsDataset(data_dir, transform=train_transform)
val_full_dataset = OfficeObjectsDataset(data_dir, transform=val_transform)


indices = list(range(len(train_full_dataset)))
train_indices, val_indices = train_test_split(
    indices, test_size=0.2, stratify=train_full_dataset.labels, random_state=42
)


train_dataset = Subset(train_full_dataset, train_indices)
val_dataset = Subset(val_full_dataset, val_indices)


num_classes = len(train_full_dataset.classes)
print(f"Number of classes in Office Objects dataset: {num_classes}")


num_workers = 2
train_dl = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=num_workers, pin_memory=torch.cuda.is_available())
val_dl = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=num_workers, pin_memory=torch.cuda.is_available())

Dataset contents: ['dining table', 'laptop', 'keyboard', 'chair', 'person']
Number of classes in Office Objects dataset: 5


In [10]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninsta

In [ ]:


import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import time
from torchmetrics import Accuracy, Precision, Recall, F1Score


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)


for name, param in model.named_parameters():
    if "conv1" in name or "bn1" in name or "layer1" in name:
        param.requires_grad = False
    else:
        param.requires_grad = True


num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.3),  
     nn.Linear(num_ftrs, num_classes)  
)
model = model.to(device)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    [param for param in model.parameters() if param.requires_grad],
    lr=0.0005
)


top1_acc = Accuracy(task="multiclass", num_classes=num_classes, top_k=1).to(device)
top5_acc = Accuracy(task="multiclass", num_classes=num_classes, top_k=5).to(device)
precision = Precision(task="multiclass", num_classes=num_classes, average='macro').to(device)
recall = Recall(task="multiclass", num_classes=num_classes, average='macro').to(device)
f1 = F1Score(task="multiclass", num_classes=num_classes, average='macro').to(device)


def train_model(model, train_dl, val_dl, criterion, optimizer, num_epochs=15, patience=3):
    best_val_acc = 0.0
    best_model_wts = None
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

       
        model.train()
        running_loss = 0.0
        running_corrects = 0
        total = 0

        
        top1_acc.reset()
        top5_acc.reset()
        precision.reset()
        recall.reset()
        f1.reset()

        for inputs, labels in train_dl:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            running_corrects += torch.sum(preds == labels.data)
            total += inputs.size(0)

            
            top1_acc.update(outputs, labels)
            top5_acc.update(outputs, labels)
            precision.update(outputs, labels)
            recall.update(outputs, labels)
            f1.update(outputs, labels)

        epoch_loss = running_loss / total
        epoch_acc = running_corrects.double() / total
        print(f'Train Loss: {epoch_loss:.4f}')
        print(f'Train Top-1 Acc: {top1_acc.compute():.4f}')
        print(f'Train Top-5 Acc: {top5_acc.compute():.4f}')
        print(f'Train Precision: {precision.compute():.4f}')
        print(f'Train Recall: {recall.compute():.4f}')
        print(f'Train F1: {f1.compute():.4f}')

        
        model.eval()
        val_loss = 0.0
        val_corrects = 0
        val_total = 0

       
        top1_acc.reset()
        top5_acc.reset()
        precision.reset()
        recall.reset()
        f1.reset()

        with torch.no_grad():
            for inputs, labels in val_dl:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels.data)
                val_total += inputs.size(0)

               
                top1_acc.update(outputs, labels)
                top5_acc.update(outputs, labels)
                precision.update(outputs, labels)
                recall.update(outputs, labels)
                f1.update(outputs, labels)

        val_loss = val_loss / val_total
        val_acc = val_corrects.double() / val_total
        print(f'Val Loss: {val_loss:.4f}')
        print(f'Val Top-1 Acc: {top1_acc.compute():.4f}')
        print(f'Val Top-5 Acc: {top5_acc.compute():.4f}')
        print(f'Val Precision: {precision.compute():.4f}')
        print(f'Val Recall: {recall.compute():.4f}')
        print(f'Val F1: {f1.compute():.4f}')

        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f'Early stopping after {epoch+1} epochs')
            model.load_state_dict(best_model_wts)
            break

        print()

    return model


model = train_model(model, train_dl, val_dl, criterion, optimizer, num_epochs=10, patience=3)


torch.save(model.state_dict(), 'office_objects_model.pth')
print("Model saved as 'office_objects_model.pth'")

Using device: cuda
Epoch 1/10
----------
Train Loss: 1.0411
Train Top-1 Acc: 0.5826
Train Top-5 Acc: 1.0000
Train Precision: 0.4509
Train Recall: 0.3796
Train F1: 0.3971
Val Loss: 0.9721
Val Top-1 Acc: 0.6079
Val Top-5 Acc: 1.0000
Val Precision: 0.4720
Val Recall: 0.4328
Val F1: 0.4439

Epoch 2/10
----------
Train Loss: 0.9686
Train Top-1 Acc: 0.6046
Train Top-5 Acc: 1.0000
Train Precision: 0.4840
Train Recall: 0.4155
Train F1: 0.4359
Val Loss: 0.9410
Val Top-1 Acc: 0.6160
Val Top-5 Acc: 1.0000
Val Precision: 0.5018
Val Recall: 0.4385
Val F1: 0.4263

Epoch 3/10
----------
Train Loss: 0.9423
Train Top-1 Acc: 0.6134
Train Top-5 Acc: 1.0000
Train Precision: 0.4958
Train Recall: 0.4355
Train F1: 0.4552
Val Loss: 0.9415
Val Top-1 Acc: 0.6122
Val Top-5 Acc: 1.0000
Val Precision: 0.4954
Val Recall: 0.4569
Val F1: 0.4531

Epoch 4/10
----------
Train Loss: 0.9218
Train Top-1 Acc: 0.6206
Train Top-5 Acc: 1.0000
Train Precision: 0.5048
Train Recall: 0.4442
Train F1: 0.4645
Val Loss: 0.9536
Val To

In [ ]:


import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import time
from torchmetrics import Accuracy, Precision, Recall, F1Score


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)


for name, param in model.named_parameters():
        param.requires_grad = True


num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.3),  
     nn.Linear(num_ftrs, num_classes)  
)
model = model.to(device)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    [param for param in model.parameters() if param.requires_grad],
    lr=0.0001
)


top1_acc = Accuracy(task="multiclass", num_classes=num_classes, top_k=1).to(device)
top5_acc = Accuracy(task="multiclass", num_classes=num_classes, top_k=5).to(device)
precision = Precision(task="multiclass", num_classes=num_classes, average='macro').to(device)
recall = Recall(task="multiclass", num_classes=num_classes, average='macro').to(device)
f1 = F1Score(task="multiclass", num_classes=num_classes, average='macro').to(device)


def train_model(model, train_dl, val_dl, criterion, optimizer, num_epochs=15, patience=3):
    best_val_acc = 0.0
    best_model_wts = None
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        
        model.train()
        running_loss = 0.0
        running_corrects = 0
        total = 0

        
        top1_acc.reset()
        top5_acc.reset()
        precision.reset()
        recall.reset()
        f1.reset()

        for inputs, labels in train_dl:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            running_corrects += torch.sum(preds == labels.data)
            total += inputs.size(0)

            
            top1_acc.update(outputs, labels)
            top5_acc.update(outputs, labels)
            precision.update(outputs, labels)
            recall.update(outputs, labels)
            f1.update(outputs, labels)

        epoch_loss = running_loss / total
        epoch_acc = running_corrects.double() / total
        print(f'Train Loss: {epoch_loss:.4f}')
        print(f'Train Top-1 Acc: {top1_acc.compute():.4f}')
        print(f'Train Top-5 Acc: {top5_acc.compute():.4f}')
        print(f'Train Precision: {precision.compute():.4f}')
        print(f'Train Recall: {recall.compute():.4f}')
        print(f'Train F1: {f1.compute():.4f}')

        
        model.eval()
        val_loss = 0.0
        val_corrects = 0
        val_total = 0

        
        top1_acc.reset()
        top5_acc.reset()
        precision.reset()
        recall.reset()
        f1.reset()

        with torch.no_grad():
            for inputs, labels in val_dl:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels.data)
                val_total += inputs.size(0)

                
                top1_acc.update(outputs, labels)
                top5_acc.update(outputs, labels)
                precision.update(outputs, labels)
                recall.update(outputs, labels)
                f1.update(outputs, labels)

        val_loss = val_loss / val_total
        val_acc = val_corrects.double() / val_total
        print(f'Val Loss: {val_loss:.4f}')
        print(f'Val Top-1 Acc: {top1_acc.compute():.4f}')
        print(f'Val Top-5 Acc: {top5_acc.compute():.4f}')
        print(f'Val Precision: {precision.compute():.4f}')
        print(f'Val Recall: {recall.compute():.4f}')
        print(f'Val F1: {f1.compute():.4f}')

       
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f'Early stopping after {epoch+1} epochs')
            model.load_state_dict(best_model_wts)
            break

        print()

    return model


model = train_model(model, train_dl, val_dl, criterion, optimizer, num_epochs=15, patience=3)


torch.save(model.state_dict(), 'office_objects_model.pth')
print("Model saved as 'office_objects_model.pth'")

Using device: cuda
Epoch 1/15
----------
Train Loss: 1.0763
Train Top-1 Acc: 0.5686
Train Top-5 Acc: 1.0000
Train Precision: 0.4244
Train Recall: 0.3705
Train F1: 0.3851
Val Loss: 0.9653
Val Top-1 Acc: 0.6068
Val Top-5 Acc: 1.0000
Val Precision: 0.4859
Val Recall: 0.3980
Val F1: 0.4175

Epoch 2/15
----------
Train Loss: 0.9684
Train Top-1 Acc: 0.6039
Train Top-5 Acc: 1.0000
Train Precision: 0.4770
Train Recall: 0.4149
Train F1: 0.4340
Val Loss: 0.9388
Val Top-1 Acc: 0.6207
Val Top-5 Acc: 1.0000
Val Precision: 0.5088
Val Recall: 0.4505
Val F1: 0.4654

Epoch 3/15
----------
Train Loss: 0.9322
Train Top-1 Acc: 0.6159
Train Top-5 Acc: 1.0000
Train Precision: 0.4975
Train Recall: 0.4382
Train F1: 0.4576
Val Loss: 0.9356
Val Top-1 Acc: 0.6177
Val Top-5 Acc: 1.0000
Val Precision: 0.4986
Val Recall: 0.4556
Val F1: 0.4720

Epoch 4/15
----------
Train Loss: 0.9090
Train Top-1 Acc: 0.6232
Train Top-5 Acc: 1.0000
Train Precision: 0.5082
Train Recall: 0.4524
Train F1: 0.4717
Val Loss: 0.9278
Val To